# Election windows vs. message length — stage 4 chart (own custom analysis)

Builds the four stage-4 bar charts designed in `analysis-log.md` (Analysis 3), for the
stage-1 proposition:

> Average message length (characters) is higher inside election windows than outside
> them.

Loads the already-anonymised, already-featured export from
[`01.3-your-own-chat.ipynb`](../lesson1/01.3-your-own-chat.ipynb) — no re-parsing here,
this notebook only adds the election-window flag, the `n_chars` / `is_media` features,
and the four stage-4 charts on top.

In [ ]:
import pandas as pd

from goad_toolkit.datatransforms import FlagDates, GroupAgg, Pipeline, RegexFeature
from goad_toolkit.visualizer import GroupedBarPlot, HighlightCategory, PlotSettings

from scripts.plots import BarPlot
from wa_analyzer.data import load_own_chat

own = load_own_chat()
own.shape

## Reference periods

Election days copied from `analysis-log.md` stage 2 (analysis 1's reference-period
table) — the same static, approximate ±30-day window around each date, not recomputed
here.

In [ ]:
ELECTION_DAYS = [
    ("US presidential 2020", "2020-11-03"),
    ("NL general 2021", "2021-03-17"),
    ("NL general (snap) 2023", "2023-11-22"),
    ("US presidential 2024", "2024-11-05"),
    ("NL general 2025", "2025-10-29"),
]
WINDOW = pd.Timedelta(days=30)

# Stage 2's decision: reuse the same election-window join pattern as the lockdown flag
# in analysis 1 -- FlagDates over the union of every day inside any window.
election_dates = pd.concat([
    pd.Series(pd.date_range(pd.Timestamp(day) - WINDOW, pd.Timestamp(day) + WINDOW))
    for _, day in ELECTION_DAYS
])

own = Pipeline().add(
    FlagDates, column="timestamp", dates=election_dates, feature="is_election_period"
).apply(own)

own["election_period"] = own["is_election_period"].map(
    {True: "election window", False: "rest of the chat"}
)
own["election_period"].value_counts()

## New features: `n_chars`, `is_media`

Stage 2 decisions: character count as the length metric, and excluding media
placeholders (`<Media weggelaten>`) before computing any length statistic -- the
placeholder text isn't a real message length.

In [ ]:
own = Pipeline().add(
    RegexFeature, name="media", column="message",
    pattern=r"<Media weggelaten>", feature="is_media", mode="has",
).apply(own)

own["n_chars"] = own["message"].str.len()

text_only = own.loc[~own["is_media"]].copy()
print(f"{own['is_media'].sum():,} media placeholders excluded of {len(own):,} messages")
text_only[["author", "election_period", "n_chars"]].head()

## Pooled comparison

Stage 3 decision: report both mean and median, since a long tail of genuinely long
messages could pull the mean around while the median stays resistant -- two separate
bar charts rather than picking one number.

In [ ]:
pooled = (
    text_only.groupby("election_period")["n_chars"]
    .agg(["mean", "median", "size"])
    .reset_index()
)
pooled

In [ ]:
order = ["rest of the chat", "election window"]

mean_settings = PlotSettings(
    figsize=(7, 4),
    title="Average message length: election window vs. rest of the chat",
    xlabel="mean characters per message",
    ylabel="",
    highlight=["election window"],
)
bars = BarPlot(mean_settings)
bars.plot(data=pooled, x="mean", y="election_period", order=order, color="#cccccc")
bars.plot_on(HighlightCategory(mean_settings), axis="y")

In [ ]:
median_settings = PlotSettings(
    figsize=(7, 4),
    title="Median message length: election window vs. rest of the chat",
    xlabel="median characters per message",
    ylabel="",
    highlight=["election window"],
)
bars = BarPlot(median_settings)
bars.plot(data=pooled, x="median", y="election_period", order=order, color="#cccccc")
bars.plot_on(HighlightCategory(median_settings), axis="y")

## Per-author comparison

Stage 2/3 decision: the per-author breakdown is built from the start, not bolted on
after -- it's the check on the stage-1 boring-result guard (one author driving the
whole group average) and on the stage-3 composition-shift confounder (some authors
posting *more*, not just *longer*, during elections).

In [ ]:
per_author = (
    text_only.groupby(["author", "election_period"])["n_chars"]
    .agg(["mean", "median"])
    .reset_index()
)
counts = (
    text_only.groupby(["author", "election_period"]).size()
    .reset_index(name="messages")
)
per_author = per_author.merge(counts, on=["author", "election_period"])

# Ordered by each author's overall volume, same convention as analysis 1/2 --
# same person in the same row across both charts below.
author_order = text_only.groupby("author").size().sort_values(ascending=False).index

per_author.pivot(index="author", columns="election_period", values="messages").loc[author_order]

**Context for the composition-shift confounder (stage 3), read alongside the
message counts above, not a fifth chart:** if a given author's *messages* column jumps
a lot between the two windows while their length barely moves, that person is likely
posting more during elections rather than writing longer messages -- worth checking by
eye before trusting the pooled result.

In [ ]:
author_mean_settings = PlotSettings(
    figsize=(9, 6),
    title="Average message length per author: election window vs. rest",
    xlabel="mean characters per message",
    ylabel="",
    legend_title="period",
)
GroupedBarPlot(author_mean_settings).plot(
    data=per_author, x="mean", y="author", hue="election_period",
    order=author_order, hue_order=order,
)

In [ ]:
author_median_settings = PlotSettings(
    figsize=(9, 6),
    title="Median message length per author: election window vs. rest",
    xlabel="median characters per message",
    ylabel="",
    legend_title="period",
)
GroupedBarPlot(author_median_settings).plot(
    data=per_author, x="median", y="author", hue="election_period",
    order=author_order, hue_order=order,
)

## Composition check: messages per week per author

Requested follow-up, not one of the four stage-4 charts: message *rate* (messages per
week) per author, election window vs. rest -- the direct check on the stage-3/stage-6
composition-shift confounder (does an author post more often during elections, not
just longer, than they otherwise do?). Rate rather than a raw count, since the two
windows cover very different numbers of calendar days.

In [ ]:
# Calendar-day denominator per label, from the actual export span -- not just message
# days, so a quiet week counts as zero rather than being invisible to the rate.
calendar = pd.DataFrame({
    "day": pd.date_range(
        pd.to_datetime(own["timestamp"]).min().tz_localize(None).normalize(),
        pd.to_datetime(own["timestamp"]).max().tz_localize(None).normalize(),
        freq="D",
    )
})
calendar = Pipeline().add(
    FlagDates, column="day", dates=election_dates, feature="is_election_period"
).apply(calendar)
calendar["election_period"] = calendar["is_election_period"].map(
    {True: "election window", False: "rest of the chat"}
)
weeks_per_label = calendar["election_period"].value_counts() / 7
weeks_per_label

In [ ]:
rate = (
    text_only.groupby(["author", "election_period"]).size()
    .rename("messages")
    .reset_index()
)
rate["weeks"] = rate["election_period"].map(weeks_per_label)
rate["messages_per_week"] = rate["messages"] / rate["weeks"]

rate.pivot(index="author", columns="election_period", values="messages_per_week").loc[author_order].round(1)

In [ ]:
rate_settings = PlotSettings(
    figsize=(9, 6),
    title="Messages per week per author: election window vs. rest",
    xlabel="messages per week",
    ylabel="",
    legend_title="period",
)
GroupedBarPlot(rate_settings).plot(
    data=rate, x="messages_per_week", y="author", hue="election_period",
    order=author_order, hue_order=order,
)

**Reading this alongside the length charts above:** if `pliable-tiger`'s
messages-per-week also jumps between the two bars here, the earlier length effect is
better explained as "posts more, including some longer messages, when there's more to
talk about" (a volume/composition story) than as "writes longer messages
specifically." If the rate barely moves for them, that would point back toward a
genuine per-message length change instead.

## Next: stage 5 (critique)

Look at all four charts above — away and back, the way `goad_analysis_checklist`'s
stage-5 table asks — before the next chat message answers its questions. That
interview needs your own read of the pictures, not a description of what the code
above was trying to do. Worth looking at specifically:

- Does the pooled mean and the pooled median agree, or does one show an effect the
  other doesn't (stage 3's long-tail risk)?
- In the per-author charts: is the election-window bar taller than the rest-of-chat
  bar for *most* of the 9 people, or is it a couple of people carrying the whole
  pooled result (stage 1's boring-result guard)?
- Cross-check against the message-count table above: does the length shift track with
  a volume shift for the same people (the composition-shift confounder from stage 3)?

---

# Analysis 4 — is group message rate itself higher during elections?

New goad cycle (see `analysis-log.md`), prompted by the messages-per-week diagnostic
above. **Proposition:** group message rate is higher during election windows, and
that increase is concentrated in the two already-high-volume authors
(`pliable-tiger`, `rib-tickling-curlew`) rather than broad-based. Origin is honestly
data-suggested (found while diagnosing analysis 3), so this needs an independent
check, not just a second look at the same chart -- hence the per-election breakdown
below.

Provenance decision (stage 2): counts here **include** media-placeholder messages,
matching analyses 1/2's precedent -- unlike analysis 3's rate diagnostic, which
excluded them only because it was checking a length metric.

In [ ]:
# election_name: 6-level categorical (5 named elections, or "rest of the chat").
# The 5 +-30-day windows don't overlap, so a clean categorical assignment is safe.
calendar_days = pd.date_range(
    pd.to_datetime(own["timestamp"]).min().tz_localize(None).normalize(),
    pd.to_datetime(own["timestamp"]).max().tz_localize(None).normalize(),
    freq="D",
)
election_name_by_day = pd.Series("rest of the chat", index=calendar_days)
for label, day in ELECTION_DAYS:
    window = pd.date_range(pd.Timestamp(day) - WINDOW, pd.Timestamp(day) + WINDOW)
    election_name_by_day.loc[election_name_by_day.index.isin(window)] = label

own["day"] = pd.to_datetime(own["timestamp"]).dt.tz_localize(None).dt.normalize()
own["election_name"] = own["day"].map(election_name_by_day)

TOP_VOLUME_AUTHORS = ["pliable-tiger", "rib-tickling-curlew"]
own["is_top_volume_author"] = own["author"].isin(TOP_VOLUME_AUTHORS)

own["election_name"].value_counts()

## Chart 1: pooled rate per individual election

Stage 3 decision: pooled across all 9 authors, not a per-author x per-election
matrix -- a full breakdown would be too sparse given each window is only ~60 days.
Weeks-per-label denominator built the same way as the earlier messages-per-week
check.

In [ ]:
election_label_days = calendar_days.to_series().groupby(election_name_by_day).size()
weeks_per_election = election_label_days / 7

by_election = own.groupby("election_name").size().rename("messages").reset_index()
by_election["weeks"] = by_election["election_name"].map(weeks_per_election)
by_election["messages_per_week"] = by_election["messages"] / by_election["weeks"]

election_order = [label for label, _ in ELECTION_DAYS] + ["rest of the chat"]
by_election = by_election.set_index("election_name").loc[election_order].reset_index()
by_election

In [ ]:
per_election_settings = PlotSettings(
    figsize=(9, 5),
    title="Group message rate per election vs. rest of the chat",
    xlabel="messages per week (all 9 authors)",
    ylabel="",
    highlight=[label for label, _ in ELECTION_DAYS],
)
bars = BarPlot(per_election_settings)
fig, ax = bars.plot(
    data=by_election, x="messages_per_week", y="election_name",
    order=election_order, color="#cccccc",
)
bars.plot_on(HighlightCategory(per_election_settings), axis="y")
fig.tight_layout()
fig.savefig("rate-per-election.png", dpi=200, bbox_inches="tight")

## Chart 2: with vs. without the two high-volume authors

Stage 1's exclusion falsification criterion, directly: does the election-window rate
increase survive once `pliable-tiger` and `rib-tickling-curlew` are taken out?

In [ ]:
exclusion = (
    own.groupby(["election_period", "is_top_volume_author"]).size()
    .rename("messages").reset_index()
)
exclusion["author_group"] = exclusion["is_top_volume_author"].map(
    {True: "top-2 authors only", False: "excluding top-2 authors"}
)

# Weeks-per-label already computed above (weeks_per_label, from the earlier check) --
# same calendar denominator, reused rather than recomputed.
exclusion["weeks"] = exclusion["election_period"].map(weeks_per_label)
exclusion["messages_per_week"] = exclusion["messages"] / exclusion["weeks"]
exclusion[["election_period", "author_group", "messages_per_week"]]

In [ ]:
exclusion_settings = PlotSettings(
    figsize=(9, 4),
    title="Election-window rate increase: with vs. without the top-2 authors",
    xlabel="messages per week",
    ylabel="",
    legend_title="",
)
fig, ax = GroupedBarPlot(exclusion_settings).plot(
    data=exclusion, x="messages_per_week", y="author_group", hue="election_period",
    order=["excluding top-2 authors", "top-2 authors only"], hue_order=order,
)
fig.tight_layout()
fig.savefig("rate-exclusion-check.png", dpi=200, bbox_inches="tight")

## Next: stage 5 (critique)

Look at both charts above before answering goad's stage-5 questions:

- Chart 1: does the rate look higher across most/all 5 elections, or is one election
  doing all the work (the small-n=5 risk named in stage 1/3)?
- Chart 2: once the two high-volume authors are excluded, does the remaining group's
  election-window bar still sit meaningfully above their rest-of-chat bar, or does
  the gap shrink/vanish -- the stage-1 falsification criterion, read directly off
  the chart?

## Presentation draft: "two people drive the election-week bump"

A single-comparison redraw of chart 2's finding, aimed at a slide rather than a
notebook cell -- one bar per author, the *increase* in rate (election window minus
rest of the chat), ordered, with only `pliable-tiger` and `rib-tickling-curlew`
coloured. Built from the same `own` data as analysis 4's chart 2 (media messages
included, matching that chart's methodology), not the text-only diagnostic from
analysis 3.

In [ ]:
per_author_rate = (
    own.groupby(["author", "election_period"]).size()
    .rename("messages").reset_index()
)
per_author_rate["weeks"] = per_author_rate["election_period"].map(weeks_per_label)
per_author_rate["messages_per_week"] = per_author_rate["messages"] / per_author_rate["weeks"]

delta = per_author_rate.pivot(index="author", columns="election_period", values="messages_per_week")
delta["increase"] = delta["election window"] - delta["rest of the chat"]
n_messages = per_author_rate.pivot(index="author", columns="election_period", values="messages")
delta["n_election_messages"] = n_messages["election window"]
delta["n_rest_messages"] = n_messages["rest of the chat"]
delta = delta.sort_values("increase", ascending=False).reset_index()
delta

In [ ]:
slide_settings = PlotSettings(
    figsize=(8, 5),
    title="Election windows create attractors and detractors in group chat activity",
    xlabel="increase in messages/week during election windows",
    ylabel=None,
)
bars = BarPlot(slide_settings)
fig, ax = bars.plot(
    data=delta, x="increase", y="author",
    order=delta["author"], color="#cccccc",
)
ax.set_ylabel("")  # seaborn relabels from the column name at draw time -- override after

# Colour carries data here (attractor vs. detractor), not a pre-picked identity --
# every bar is coloured by its own sign, none held back as "the boring rest".
ATTRACTOR_COLOR = "steelblue"
DETRACTOR_COLOR = "crimson"
increase_by_author = delta.set_index("author")["increase"]

ticks = ax.get_yticks()
labels = [label.get_text() for label in ax.get_yticklabels()]
for patch in ax.patches:
    centre = patch.get_y() + patch.get_height() / 2
    nearest = min(range(len(ticks)), key=lambda i: abs(ticks[i] - centre))
    author = labels[nearest]
    patch.set_facecolor(ATTRACTOR_COLOR if increase_by_author[author] >= 0 else DETRACTOR_COLOR)

# Direct value labels instead of gridlines -- the number that matters sits on the bar
# it belongs to rather than requiring the reader to read off an axis.
for patch in ax.patches:
    width = patch.get_width()
    ax.text(
        width + (0.05 if width >= 0 else -0.05),
        patch.get_y() + patch.get_height() / 2,
        f"{width:+.1f}",
        va="center",
        ha="left" if width >= 0 else "right",
        fontsize=9,
    )

from matplotlib.patches import Patch
ax.legend(
    handles=[
        Patch(facecolor=ATTRACTOR_COLOR, label="attractor -- posts more"),
        Patch(facecolor=DETRACTOR_COLOR, label="detractor -- posts less"),
    ],
    loc="lower right", framealpha=1, fontsize=9,
)

ax.margins(x=0.15)
ax.axvline(0, color="black", linewidth=0.8)
fig.tight_layout()
fig.savefig("slide-two-authors-drive-it.png", dpi=200, bbox_inches="tight")

## Presentation draft v2: percent change, not absolute

Same underlying comparison, but relative to each author's own baseline -- a +3.9
messages/week jump means something different for someone who already writes 10/week
than for someone who writes 3/week. Threshold, decided explicitly rather than left
implicit: **+-20% change** counts as a large attractor/detractor (coloured);
anything smaller stays neutral grey.

In [ ]:
delta["pct_change"] = delta["increase"] / delta["rest of the chat"] * 100

# Poisson error propagation (item 5 of the logged feedback): each rate is a count over
# a fixed number of weeks, so SE(rate) = sqrt(count) / weeks -- not a plain mean +- SE,
# which assumes independent per-observation measurements this isn't. pct_change is a
# ratio of the two rates, so its relative variance is the sum of the two rates' own
# relative variances (standard delta-method combination for a ratio of two independent
# estimates).
se_election_rate = delta["n_election_messages"] ** 0.5 / weeks_per_label["election window"]
se_rest_rate = delta["n_rest_messages"] ** 0.5 / weeks_per_label["rest of the chat"]
relative_variance = (
    (se_election_rate / delta["election window"]) ** 2
    + (se_rest_rate / delta["rest of the chat"]) ** 2
)
ratio = delta["election window"] / delta["rest of the chat"]
delta["pct_change_se"] = 100 * ratio * relative_variance**0.5

delta_pct = delta.sort_values("pct_change", ascending=False).reset_index(drop=True)
delta_pct[["author", "increase", "pct_change", "pct_change_se"]]

In [ ]:
THRESHOLD = 20  # +-20% -- decided explicitly with the student, not left implicit

SUBTITLE = "Election windows create attractors and detractors -- relative to each author's own baseline"

pct_settings = PlotSettings(
    figsize=(9.5, 5.5),
    title=SUBTITLE,
    xlabel="% change in # of messages/week during election windows (5 windows over a 6-year period)",
    ylabel=None,
)
bars = BarPlot(pct_settings)
fig, ax = bars.plot(
    data=delta_pct, x="pct_change", y="author",
    order=delta_pct["author"], color="#cccccc",
)
ax.set_ylabel("")
# Catchy headline as the figure-level title, the precise claim demoted to an
# axis-level subtitle underneath it -- a reader gets pulled in by the first,
# then knows exactly what is being measured from the second.
ax.set_title(SUBTITLE, fontsize=10, style="italic", pad=10)
fig.suptitle(
    "How do political elections affect group chat activity per user?",
    fontsize=15, fontweight="bold", y=0.99,
)

ATTRACTOR_COLOR = "steelblue"
DETRACTOR_COLOR = "crimson"
NEUTRAL_COLOR = "#cccccc"
pct_by_author = delta_pct.set_index("author")["pct_change"]
se_by_author = delta_pct.set_index("author")["pct_change_se"]

ticks = ax.get_yticks()
labels = [label.get_text() for label in ax.get_yticklabels()]
for patch in ax.patches:
    centre = patch.get_y() + patch.get_height() / 2
    nearest = min(range(len(ticks)), key=lambda i: abs(ticks[i] - centre))
    value = pct_by_author[labels[nearest]]
    if value >= THRESHOLD:
        color = ATTRACTOR_COLOR
    elif value <= -THRESHOLD:
        color = DETRACTOR_COLOR
    else:
        color = NEUTRAL_COLOR
    patch.set_facecolor(color)

# Poisson error bars (item 5) -- drawn from the pre-computed pct_change_se column, the
# same pattern BarPlotWithError uses for already-summarised intervals, just inline here
# since this is a single-series bar chart rather than a grouped one.
for patch in ax.patches:
    centre = patch.get_y() + patch.get_height() / 2
    nearest = min(range(len(ticks)), key=lambda i: abs(ticks[i] - centre))
    author = labels[nearest]
    ax.errorbar(
        patch.get_width(), centre,
        xerr=se_by_author[author],
        fmt="none", ecolor="black", capsize=4, linewidth=1,
    )

# Label carries both the % change and the election-window sample size it rests
# on -- a reader can judge for themselves whether a small n makes a % swing shaky.
# Pushed out past the error bar's tip so it never overlaps the whisker.
n_election_by_author = delta_pct.set_index("author")["n_election_messages"]
for patch in ax.patches:
    centre = patch.get_y() + patch.get_height() / 2
    nearest = min(range(len(ticks)), key=lambda i: abs(ticks[i] - centre))
    author = labels[nearest]
    width = patch.get_width()
    se = se_by_author[author]
    n = n_election_by_author[author]
    label_x = width + se + 1 if width >= 0 else width - se - 1
    ax.text(
        label_x,
        centre,
        f"{width:+.0f}% (n={n})",
        va="center",
        ha="left" if width >= 0 else "right",
        fontsize=9,
    )

from matplotlib.patches import Patch
ax.legend(
    handles=[
        Patch(facecolor=ATTRACTOR_COLOR, label=f"large attractor (>= +{THRESHOLD}%)"),
        Patch(facecolor=NEUTRAL_COLOR, label=f"within +-{THRESHOLD}%"),
        Patch(facecolor=DETRACTOR_COLOR, label=f"large detractor (<= -{THRESHOLD}%)"),
    ],
    loc="lower right", framealpha=1, fontsize=9,
)

ax.margins(x=0.35)
ax.axvline(0, color="black", linewidth=0.8)
fig.tight_layout(rect=[0, 0, 1, 0.94])  # leaves headroom for fig.suptitle above ax.set_title
fig.savefig("slide-attractors-detractors-pct.png", dpi=200, bbox_inches="tight")

## Shuffle test (item 6 of the logged feedback): computed, not a gut estimate

Stage 6's own question, finally answered with a real simulation instead of "by eye":
*if a randomly-placed ~60-day window were dropped somewhere else in the chat's
history, how often would it produce a swing this large, in the same direction?*

Decided with the student:
- **5 random windows** (same 61-day length as a real ±30-day election window) --
  coarse (results land on fifths), matching the shuffle-test norm of "a rough
  frequency", not a fine p-value.
- **Same direction only** counts as a match -- a random window only "matches"
  `pliable-tiger`'s +38% if it also produces an increase at least that large for
  `pliable-tiger`, not any large swing in either direction.
- **Restricted to a similar era as the real elections** -- drawn from within the span
  the 5 real election windows already cover (2020-10-04 to 2025-11-28), not from
  2026's tail where no real election falls. This keeps the general activity-decline
  confound (already flagged in Analysis 4) from being the thing that decides the
  answer by itself.
- Windows are drawn **non-overlapping** with the 5 real election windows (comparing a
  random window to itself would be meaningless) and with each other.
- **Baseline kept consistent with the real result:** each random window's "rest of the
  chat" comparison excludes the 5 real election windows too, the same quiet baseline
  `pct_change` was computed against above -- not a baseline contaminated by real
  election spikes for some draws and not others.

Focus of the readout: the three people named in the feedback --
`pliable-tiger`/Thijs and `rib-tickling-curlew`/Jop (attractors), `vibrant-barracuda`/
Mark (detractor) -- though the table below covers all 9 for completeness.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)  # seeded for a reproducible notebook re-run

real_windows = [
    (pd.Timestamp(day) - WINDOW, pd.Timestamp(day) + WINDOW) for _, day in ELECTION_DAYS
]
era_start = min(start for start, _ in real_windows)
era_end = max(end for _, end in real_windows)
shuffle_window_span = 2 * WINDOW  # same 61-day length (inclusive) as a real window


def _overlaps(a_start, a_end, b_start, b_end) -> bool:
    return a_start <= b_end and b_start <= a_end


N_SHUFFLES = 5  # decided with the student
max_start_offset = (era_end - era_start - shuffle_window_span).days

shuffle_windows: list[tuple[pd.Timestamp, pd.Timestamp]] = []
while len(shuffle_windows) < N_SHUFFLES:
    start = era_start + pd.Timedelta(days=int(rng.integers(0, max_start_offset)))
    end = start + shuffle_window_span
    if any(_overlaps(start, end, *w) for w in real_windows):
        continue
    if any(_overlaps(start, end, *w) for w in shuffle_windows):
        continue
    shuffle_windows.append((start, end))

shuffle_windows

In [ ]:
# Same "quiet baseline" the real pct_change was measured against -- excludes all 5
# real election windows, not just the one shuffle window being scored.
baseline_mask = ~own["is_election_period"]
rest_of_chat_days = weeks_per_label["rest of the chat"] * 7

shuffle_records = []
for shuffle_id, (start, end) in enumerate(shuffle_windows, start=1):
    in_window = own["day"].between(start, end)
    window_days = (end - start).days + 1
    rest_days = rest_of_chat_days - window_days

    for author in author_order:
        author_mask = own["author"] == author
        n_window = (author_mask & in_window).sum()
        n_rest = (author_mask & baseline_mask & ~in_window).sum()
        rate_window = n_window / (window_days / 7)
        rate_rest = n_rest / (rest_days / 7)
        shuffle_records.append({
            "shuffle_id": shuffle_id,
            "author": author,
            "pct_change": (rate_window - rate_rest) / rate_rest * 100,
        })

shuffle_df = pd.DataFrame(shuffle_records)


def _matches_direction(real_value: float, shuffled_value: float) -> bool:
    """A shuffled draw only 'matches' if it moves at least as far, same direction."""
    return shuffled_value >= real_value if real_value >= 0 else shuffled_value <= real_value


real_pct = delta_pct.set_index("author")["pct_change"]
shuffle_df["real_pct_change"] = shuffle_df["author"].map(real_pct)
shuffle_df["matches"] = [
    _matches_direction(real, shuffled)
    for real, shuffled in zip(shuffle_df["real_pct_change"], shuffle_df["pct_change"])
]

shuffle_summary = (
    shuffle_df.groupby("author")
    .agg(
        real_pct_change=("real_pct_change", "first"),
        n_matching=("matches", "sum"),
        n_draws=("matches", "size"),
    )
    .assign(frequency=lambda d: d["n_matching"] / d["n_draws"])
    .loc[author_order]
)
shuffle_summary

## Presentation draft v3: narrowed to one person, with a real story

Not a rebuild -- a new version alongside v1/v2, same as those two were kept side by
side. Decisions made with the student, informed by the SE table and shuffle test
above:

- **Scope narrowed to `pliable-tiger` alone** (item 2/4) -- the one person the
  student can tell a real, specific story about: ~10 years studying history,
  enormously politically engaged, briefly a member of a Dutch political party, and
  known in the group for driving political-topic discussions (e.g. democracy). This
  is exactly the "still to do" gap item 3 of the logged feedback named -- turning an
  anonymous code's % change into an actual claim about a specific person's behaviour.
  Real name stays out of the notebook/chart for now -- item 3's caution about
  re-checking the audience/sharing plan before real names appear anywhere shared
  still applies, unresolved here.
- **All 9 authors stay visible, in grey** (student's explicit call) -- only
  `pliable-tiger` is coloured. This keeps the "don't imply only this one person
  changed" honesty property that motivated the original multi-colour reframe,
  without needing a 3-way threshold to get there: the reader can still see every
  author's bar, just with one clearly called out.
- **Old 3-way attractor/neutral/detractor legend dropped** (item 1/2) -- replaced
  with a 2-entry legend naming the highlighted author and explaining the error bars,
  since without the threshold legend the whiskers would otherwise go unexplained.
- **Per-bar `n=` labels dropped except on the focus bar** -- the error bar already
  carries the reliability information visually for the other 8; repeating a text
  label on every bar was part of the "too much happening in one graph" item 1
  flagged. The focus bar keeps its label (% change, n, and SE together) since it's
  the one number the chart is actually about.
- **Error bars (item 5) kept on all nine bars**, not just the focus one -- a reader
  who looks past the highlight can still see that most of the grey bars are close to
  their own zero-crossing, which is part of why they're grey.

In [ ]:
FOCUS_AUTHOR = "pliable-tiger"
FOCUS_COLOR = "steelblue"
NEUTRAL_COLOR = "#cccccc"

v3_subtitle = "% change relative to each author's own baseline -- error bars show +-1 SE (Poisson)"

v3_settings = PlotSettings(
    figsize=(9.5, 6.3),
    title=v3_subtitle,
    xlabel="% change in # of messages/week during election windows (5 windows over a 6-year period)",
    ylabel=None,
)
bars = BarPlot(v3_settings)
fig, ax = bars.plot(
    data=delta_pct, x="pct_change", y="author",
    order=delta_pct["author"], color=NEUTRAL_COLOR,
)
ax.set_ylabel("")
ax.set_title(v3_subtitle, fontsize=10, style="italic", pad=10)
fig.suptitle(
    f"The historian in the group ({FOCUS_AUTHOR}) awakes and activates group chat during election periods",
    fontsize=14, fontweight="bold", y=0.99,
)

ticks = ax.get_yticks()
labels = [label.get_text() for label in ax.get_yticklabels()]


def _author_for_patch(patch) -> str:
    centre = patch.get_y() + patch.get_height() / 2
    nearest = min(range(len(ticks)), key=lambda i: abs(ticks[i] - centre))
    return labels[nearest]


# Only the focus author is coloured -- the other 8 stay visible in grey rather than
# being dropped, per the student's call to show all the data.
for patch in ax.patches:
    author = _author_for_patch(patch)
    patch.set_facecolor(FOCUS_COLOR if author == FOCUS_AUTHOR else NEUTRAL_COLOR)

# Error bars on every bar (item 5) -- lets a reader see that most of the grey bars
# sit close to their own zero-crossing, which is part of why they're grey.
for patch in ax.patches:
    author = _author_for_patch(patch)
    ax.errorbar(
        patch.get_width(), patch.get_y() + patch.get_height() / 2,
        xerr=se_by_author[author],
        fmt="none", ecolor="black", capsize=4, linewidth=1,
    )

# Value label kept only on the focus bar -- the other 8 rely on the error bar alone,
# not a repeated text label on every bar (item 1's "too much per bar" fix).
focus_patch = next(p for p in ax.patches if _author_for_patch(p) == FOCUS_AUTHOR)
focus_width = focus_patch.get_width()
focus_se = se_by_author[FOCUS_AUTHOR]
focus_n = int(delta_pct.set_index("author").loc[FOCUS_AUTHOR, "n_election_messages"])
ax.text(
    focus_width + focus_se + 1,
    focus_patch.get_y() + focus_patch.get_height() / 2,
    f"{focus_width:+.0f}% (n={focus_n}, +-{focus_se:.0f} SE)",
    va="center", ha="left", fontsize=10, fontweight="bold",
)

# Legend now explains only the error bar -- naming the highlighted author in a legend
# was redundant once the title itself names `pliable-tiger` directly.
from matplotlib.lines import Line2D

ax.legend(
    handles=[
        Line2D([0], [0], color="black", marker="|", linestyle="none",
               markersize=10, markeredgewidth=1.5, label="error bar: +-1 SE (Poisson)"),
    ],
    loc="lower right", framealpha=1, fontsize=9,
)

ax.margins(x=0.3)
ax.axvline(0, color="black", linewidth=0.8)

# Footnote for a non-technical reader: what the error bar means and how to read it,
# spelled out on the figure itself rather than assumed knowledge.
fig.text(
    0.5, 0.015,
    "An error bar shows the range the true value could plausibly fall in, just from\n"
    "week-to-week randomness. A bar whose error whisker reaches past zero could just be\n"
    "noise; one whose whisker clears zero comfortably (like pliable-tiger's) is unlikely\n"
    "to be due to chance alone.",
    ha="center", va="bottom", fontsize=8, style="italic", color="#444444",
)

fig.tight_layout(rect=[0, 0.09, 1, 0.94])
fig.savefig("slide-pliable-tiger-election-effect.png", dpi=200, bbox_inches="tight")